# ANÁLISIS DE TENDENCIAS DE VIDEOS DE YOUTUBE — GRAN BRETAÑA (GB)
## Aplicación de la Metodología CRISP-DM
**Fundamentos de Data Science (1ACC0216) — TB2 · Ciclo 2026-1**

**Sección 16310 · Grupo 1 (GB)** | Integrantes: [Nombre 1], [Nombre 2], [Nombre 3], [Nombre 4]

### Cargar los archivos en Colab
Ejecuta esta celda PRIMERO. Se abrirá un botón; selecciona **los dos archivos juntos**: `GBvideos.csv` y `GB_category_id.json`.

> Importante: sube el `GBvideos.csv` tal como lo descargaste, **sin abrirlo en Excel** (Excel rompe el formato del CSV).

### Librerías a utilizar

In [1]:
## Tratamiento de datos
import numpy as np
import pandas as pd
import json

## Preprocesado y modelado
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Gráficos
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_style('whitegrid')

## 1. COMPRENSIÓN DEL NEGOCIO

**Objetivos del proyecto:** una empresa de marketing digital desea conocer las tendencias de los videos de YouTube en Gran Bretaña para orientar la producción de contenido, priorizar categorías con mayor alcance y aceptación, e identificar canales de referencia.

**Objetivos de Data Science:**
- **Variable dependiente (a predecir):** `views` (número de vistas), indicador directo del alcance de un video.
- **Variables independientes principales:** `likes`, `dislikes`, `comment_count`, `category_id`, y variables derivadas (`days_to_trend`, `title_len`, `n_tags`).

## 2. COMPRENSIÓN DE LOS DATOS

### Base de datos
Recolección de los datos iniciales: se cargan el CSV de videos y el JSON de categorías.

In [3]:
# Lectura del CSV
# El archivo original se lee directamente. Si al subirlo a Colab se corrompió
# y aparece un ParserError, vuelve a subir el archivo original sin abrirlo en Excel.
df = pd.read_csv('GBvideos.csv')
df.head(5)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,Jw1Y-zhQURU,17.14.11,John Lewis Christmas Ad 2017 - #MozTheMonster,John Lewis,26,2017-11-10T07:38:29.000Z,"christmas|""john lewis christmas""|""john lewis""|...",7224515.0,55681.0,10247.0,9479.0,https://i.ytimg.com/vi/Jw1Y-zhQURU/default.jpg,False,False,False,Click here to continue the story and make your...
1,3s1rvMFUweQ,17.14.11,Taylor Swift: …Ready for It? (Live) - SNL,Saturday Night Live,24,2017-11-12T06:24:44.000Z,"SNL|""Saturday Night Live""|""SNL Season 43""|""Epi...",1053632.0,25561.0,2294.0,2757.0,https://i.ytimg.com/vi/3s1rvMFUweQ/default.jpg,False,False,False,Musical guest Taylor Swift performs …Ready for...
2,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579.0,787420.0,43420.0,125882.0,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...
3,PUTEiSjKwJU,17.14.11,Goals from Salford City vs Class of 92 and Fri...,Salford City Football Club,17,2017-11-13T02:30:38.000Z,"Salford City FC|""Salford City""|""Salford""|""Clas...",27833.0,193.0,12.0,37.0,https://i.ytimg.com/vi/PUTEiSjKwJU/default.jpg,False,False,False,Salford drew 4-4 against the Class of 92 and F...
4,rHwDegptbI4,17.14.11,Dashcam captures truck's near miss with child ...,Cute Girl Videos,25,2017-11-13T01:45:13.000Z,[none],9815.0,30.0,2.0,30.0,https://i.ytimg.com/vi/rHwDegptbI4/default.jpg,False,False,False,Dashcam captures truck's near miss with child ...


In [ ]:
# Cargar el JSON de categorías y crear el diccionario id -> nombre
cats = json.load(open('GB_category_id.json'))
catmap = {int(i['id']): i['snippet']['title'] for i in cats['items']}
catmap[29] = 'Nonprofits & Activism'   # id presente en los datos pero ausente en el JSON de GB
catmap

### Descripción de los datos

In [ ]:
# Estructura de los datos (tipos de cada columna)
df.dtypes

In [ ]:
# Información general
df.info()

In [ ]:
# Cantidad de datos (filas, columnas)
df.shape

In [ ]:
# Número de videos únicos (un video aparece varios días en tendencia)
print('Videos únicos:', df.video_id.nunique())
print('Periodo:', df.trending_date.min(), '->', df.trending_date.max())

### Exploración de los datos

In [ ]:
# Mapear la categoría a su nombre legible
df['category'] = df.category_id.map(catmap)

# Tabla de frecuencia: apariciones por categoría
df['category'].value_counts()

In [ ]:
# Tabla de frecuencia: canales más frecuentes
df['channel_title'].value_counts().head(10)

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df[['views','likes','dislikes','comment_count']].describe()

#### Análisis de Correlación
Heatmap de correlación entre las variables numéricas (mismo estilo visto en clase).

In [ ]:
variables_numericas = df[['views','likes','dislikes','comment_count']]
plt.figure(figsize=(8,6))
sns.heatmap(variables_numericas.corr(), cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Correlación entre variables numéricas')
plt.show()

### Verificar la calidad de los datos

In [ ]:
# Valores nulos por columna
df.isnull().sum()

In [ ]:
# Solo la columna 'description' tiene nulos. Ninguna variable numérica clave falta.
print('Nulos en description:', df.description.isnull().sum())

## 3. PREPARACIÓN DE LOS DATOS

### Limpiar los datos
Conversión de fechas y unión con las categorías.

In [ ]:
df['trending_date'] = pd.to_datetime(df.trending_date, format='%y.%d.%m')
df['publish_time'] = pd.to_datetime(df.publish_time).dt.tz_localize(None)
df['description'] = df.description.fillna('')   # tratar faltantes
df.head(3)

### Pre-procesar los datos
Detección y tratamiento de outliers en `days_to_trend`.

In [ ]:
# Construir la variable días para volverse tendencia
df['days_to_trend'] = (df.trending_date - df.publish_time.dt.normalize()).dt.days

# Explorar outliers
print('Registros con más de 365 días:', (df.days_to_trend > 365).sum())
print('Mediana de días para ser tendencia:', df.days_to_trend.median())

### Construir nuevos datos
Derivación de nuevos atributos a partir de los existentes.

In [ ]:
df['title_len'] = df.title.str.len()          # longitud del título
df['n_tags']    = df.tags.str.count(r'\|') + 1  # número de etiquetas
df['pub_hour']  = df.publish_time.dt.hour       # hora de publicación

# Conjunto por video único (última aparición = mayor acumulado) para métricas por categoría
uni = df.sort_values('trending_date').drop_duplicates('video_id', keep='last')

# Guardar el dataset limpio y enriquecido
df.to_csv('GBvideos_clean.csv', index=False)
print('Dataset limpio guardado. Filas:', len(df))

### Requerimientos
Respuesta a las preguntas del cliente, cada una con su visualización.

**Q1. ¿Qué categorías son las de mayor tendencia?**

In [ ]:
q1 = df['category'].value_counts().head(8)
print(q1)
q1.sort_values().plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Q1 - Categorías con mayor tendencia'); plt.xlabel('Nº de apariciones')
plt.tight_layout(); plt.show()

**Q2. ¿Qué categorías gustan más y cuáles menos?** (promedio de likes por video único)

In [ ]:
q2 = uni.groupby('category').likes.mean().sort_values(ascending=False)
print('Más gustan:'); print(q2.head(4).round(0))
print('\nMenos gustan:'); print(q2.tail(3).round(0))
q2.sort_values().tail(8).plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Q2 - Likes promedio por categoría'); plt.xlabel('Likes promedio')
plt.tight_layout(); plt.show()

**Q3. ¿Qué categorías tienen la mejor proporción Me gusta / No me gusta?**

In [ ]:
g = uni.groupby('category').agg(likes=('likes','sum'), dislikes=('dislikes','sum'))
g['ratio'] = g.likes / g.dislikes
print(g.ratio.sort_values(ascending=False).head(5).round(1))
g.ratio.sort_values().tail(8).plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Q3 - Ratio Me gusta / No me gusta'); plt.xlabel('Likes por cada dislike')
plt.tight_layout(); plt.show()

**Q4. ¿Qué categorías tienen la mejor proporción Vistas / Comentarios?**

In [ ]:
g2 = uni.groupby('category').agg(views=('views','sum'), comments=('comment_count','sum'))
g2['v_per_c'] = g2.views / g2.comments
print(g2.v_per_c.sort_values(ascending=False).head(5).round(0))
g2.v_per_c.sort_values().tail(8).plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Q4 - Ratio Vistas / Comentarios'); plt.xlabel('Vistas por comentario')
plt.tight_layout(); plt.show()

**Q5. ¿Cómo ha cambiado el volumen de videos en tendencia a lo largo del tiempo?**

In [ ]:
ts = df.groupby('trending_date').size()
ts.plot(color='#C8102E', figsize=(9,4))
plt.title('Q5 - Volumen diario de videos en tendencia'); plt.ylabel('Nº de videos')
plt.tight_layout(); plt.show()

**Q6. ¿Qué canales son tendencia con mayor y menor frecuencia?**

In [ ]:
q6 = df['channel_title'].value_counts()
print('Más frecuentes:'); print(q6.head(6))
print('\nCanales con una sola aparición:', (q6 == 1).sum())
q6.head(10).sort_values().plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Q6 - Canales más frecuentes en tendencia'); plt.xlabel('Nº de apariciones')
plt.tight_layout(); plt.show()

**Q7. ¿En qué Estados se presentan más vistas, likes y dislikes?**

> **No resoluble con los datos entregados:** el archivo `GBvideos.csv` no contiene las columnas geográficas `state`, `lat`, `lon`. Procedimiento listo para cuando estén disponibles:
```python
# geo = df.groupby('state').agg(views=('views','sum'), likes=('likes','sum'), dislikes=('dislikes','sum'))
# import plotly.express as px
# px.scatter_geo(geo.reset_index(), lat='lat', lon='lon', size='likes')
```

**Q8. ¿Los videos en tendencia reciben más comentarios positivos?**

> El dataset solo contiene `comment_count` (cantidad), no el texto de los comentarios, por lo que no es posible medir el sentimiento/positividad directamente.

## 4. MODELADO Y EVALUACIÓN DE RESULTADOS
**Q9. ¿Es factible predecir el número de vistas?**

### Definición de Variables
Variable dependiente (Y) e independientes (X), siguiendo el estilo de clase.

In [ ]:
# Filtrar outliers de days_to_trend solo para el modelo
datos = df[df.days_to_trend.between(0, 365)].copy()

# Variables independientes (X) y dependiente (Y)
X = datos[['likes','dislikes','comment_count','days_to_trend','title_len','n_tags','pub_hour','category_id']]
Y = np.log1p(datos['views'])   # log-transform para reducir el sesgo de las vistas
print('X shape:', X.shape, '| Y shape:', Y.shape)

### Generar el plan de prueba
División en conjunto de entrenamiento y prueba (80/20).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1280)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

### Construir el modelo
#### Modelo 1: Regresión Lineal (modelo base visto en clase)

In [ ]:
# Definimos el algoritmo a utilizar
lr_simple = LinearRegression()

# Entrenamos el modelo
lr_simple.fit(X_train, y_train)

# Estimadores de los coeficientes
print('DATOS DEL MODELO REGRESIÓN LINEAL')
print('Pendientes (coef):', lr_simple.coef_.round(4))
print('Interceptor:', round(lr_simple.intercept_, 4))

In [ ]:
# Predicción del modelo
Y_pred_lr = lr_simple.predict(X_test)
Y_pred_lr[:5]

#### Modelo 2: Random Forest (modelo no lineal, para comparar)

In [ ]:
rf = RandomForestRegressor(n_estimators=120, max_depth=18, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
Y_pred_rf = rf.predict(X_test)
Y_pred_rf[:5]

### Evaluar el modelo
Métricas de rendimiento sobre el conjunto de prueba.

In [ ]:
print('=== REGRESIÓN LINEAL ===')
print('R2 :', round(r2_score(y_test, Y_pred_lr), 3))
print('MAE:', round(mean_absolute_error(y_test, Y_pred_lr), 3))
print()
print('=== RANDOM FOREST ===')
print('R2 :', round(r2_score(y_test, Y_pred_rf), 3))
print('MAE:', round(mean_absolute_error(y_test, Y_pred_rf), 3))

Comparación gráfica: valores reales vs. predichos (Random Forest).

In [ ]:
df_pred = pd.DataFrame({'Real': y_test, 'Pred': Y_pred_rf})
sns.scatterplot(data=df_pred, x='Real', y='Pred', alpha=0.3, color='#C8102E')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.title('Valores reales vs. predichos (log vistas)'); plt.tight_layout(); plt.show()

Importancia de las variables en el Random Forest.

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(color='#C8102E', figsize=(8,4))
plt.title('Importancia de variables - Random Forest'); plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))